In [1]:
"""
Text preprocessing pipeline for Singapore political Reddit data.
Designed for sentiment analysis with scikit-learn or HuggingFace models.

Pipeline steps (in order):
  1. Lowercasing & URL/mention removal
  2. Filler word removal  (lah, lor, leh ...)
  3. Slang / microtext normalization  (sgp -> singapore)
  4. Entity / alias normalization  (PAP -> ent_peoples_action_party)
  5. Phrase normalization  (cost of living -> cost_of_living)
  6. Negation handling  (can't good -> not good)
  7. Optional lowercasing / light lemmatization
  8. Entity token restoration  (ent_peoples_action_party -> peoples action party)
"""

from pathlib import Path
import sys
import pandas as pd


def find_repo_root(start_dir: Path) -> Path:
    """Find project root by locating the expected raw input file path."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "data" / "raw" / "data_annotation.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/data_annotation.csv from the current working directory."
    )


repo_root = find_repo_root(Path.cwd())

preprocess_dir = repo_root / "classification" / "svm"
if not (preprocess_dir / "preprocess_pipeline.py").exists():
    raise FileNotFoundError(f"Missing preprocess module at: {preprocess_dir / 'preprocess_pipeline.py'}")
if str(preprocess_dir) not in sys.path:
    sys.path.insert(0, str(preprocess_dir))

from preprocess_pipeline import preprocess_text

TEXT_COL = "body_training"
INPUT_PATH = repo_root / "data" / "raw" / "data_annotation.csv"
OUTPUT_PATH = repo_root / "data" / "cleaned" / "data_annotation_clean.csv"


df = pd.read_csv(INPUT_PATH)
if TEXT_COL not in df.columns:
    raise ValueError(f"Missing required column: {TEXT_COL}")

df[TEXT_COL] = df[TEXT_COL].astype("string")
df["clean_text"] = df[TEXT_COL].map(preprocess_text)

print(df[[TEXT_COL, "clean_text"]].head(10))
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved -> {OUTPUT_PATH}")

                                       body_training  \
0  WP won't take mosquitos. They worked so hard t...   
1  Unless WP wants to take the mosquitoes under t...   
2  It's not a given for WP to win the by-election...   
3  I think SDP is too far to the left for fiscall...   
4  Maybe have realistic/concrete goals of what th...   
5  Hmm. I feel that it's too early to tell. If LE...   
6  Singapore’s population is too small to have to...   
7  Yeah, so step 1 should be attracting credible ...   
8  SDP and WP don’t need to work together at all ...   
9  WP has an eastern strategy while SDP has a nor...   

                                          clean_text  
0  workers party not take mosquitos. they worked ...  
1  unless workers party wants to take the mosquit...  
2  it's not a given for workers party to win the ...  
3  i think singapore democratic party is too far ...  
4  maybe have realistic / concrete goals of what ...  
5  hmm. i feel that it's too early to tell. if is... 